# 10 — ACDC / MRzero: optimising the phase at every echo, on a numerical phantom

Generation 08 chooses the ACDC currents by comparing simulated **images**.
Generation 10 tests another idea: choose them by looking only at the **phase** of the object at the echo of every EPI line.

**The idea.** At the echo of line $n$ (time $T_n$ after the RF pulse), a point $p$ of the object has accumulated the phase

$$\varphi_n(p) \;=\; \underbrace{2\pi\,B_0(p)\,T_n}_{\beta_n(p)} \;+\; \sum_{i=1}^{n} A(p)\,x_i ,$$

where $x_i$ is the vector of currents given during line $i$ and $A(p)\,x_i$ is the phase that these currents add at $p$.
The currents are chosen to minimise

$$\Psi \;=\; \sum_n \lVert\varphi_n\rVert^2 , \qquad \lVert\varphi_n\rVert^2 \;=\; \frac{1}{P}\sum_p \varphi_n^2(p) .$$

$\Psi$ is quadratic in the currents and the hardware limits are convex: there is a single minimum, found in a few seconds, without any simulation.

**What is kept from generation 08.** The signal model (MRzero gives the magnetisation at each voxel centre; one voxel spans the whole slice; signal loss inside the voxel along x, y and z), one current state per EPI line, the amplifier response and the hardware limits.

**What this notebook does.**

1. builds a numerical head phantom placed in the ACDC calibration;
2. simulates its EPI image with MRzero, without ACDC;
3. minimises $\Psi$ (as written above, and with two options explained in section 4);
4. simulates the image obtained with each set of currents and compares it with a static shim and with the image objective of generation 08.

The functions are in this folder; only `data/ACDC_3T.npz` is read from the parent folder. Restart the kernel before switching between generations.

In [ ]:
from datetime import datetime
from pathlib import Path
import json, sys, time

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd()
if not (HERE / "phase_optimization.py").is_file():
    raise FileNotFoundError("Open this notebook from the MRzero_generation_10 folder.")
sys.path.insert(0, str(HERE))

from phantom import make_head_phantom
from epi_sequence import make_epi_sequence
from acdc_calibration import load_calibration, sample_coil_fields
from acdc_hardware import make_hardware, check_limits, quantize_to_dac
from epi_simulation import EpiSimulator
from phase_optimization import PhaseProblem, optimize_phase, static_shim
from image_optimization import optimize_image, relative_image_error
from display_helpers import METHOD_COLORS, use_notebook_style, table, image_row

use_notebook_style()
CALIBRATION_PATH = HERE.parent / "data" / "ACDC_3T.npz"
OUTPUT_DIR = HERE / "results" / datetime.now().strftime("mrzero10_%Y%m%d_%H%M%S")
OUTPUT_DIR.mkdir(parents=True)
print("Calibration:", CALIBRATION_PATH)
print("Results:", OUTPUT_DIR)

## 1. Parameters to change

The sequence and hardware values are those of generation 08 and of the scan `gre_epi_128x128_fov200mm_..._v6.seq`.

**Options of the phase objective** (section 4 explains them):

- `INTRAVOXEL_WEIGHT`: weight of the phase variation *inside* each voxel. `0` uses only the voxel centres; `1/12` is the exact mean of $\varphi^2$ over the voxel volume.
- `SIGNAL_THRESHOLD`: voxels that keep less than this fraction of their signal at TE without correction are left out of the mean.

**Comparison with generation 08**: `RUN_IMAGE_OPTIMIZATION=False` skips the image objective (the slowest step).

In [ ]:
# Phantom and EPI sequence.
MATRIX = 128                    # image and phantom matrix
FOV_M = 0.200
SLICE_THICKNESS_M = 0.005
SLICE_Z_M = 0.030               # position of the slice in the coil frame
B0_SCALE = 1.0                  # multiplies the off-resonance of the phantom (0: perfect magnet)
ECHO_SPACING_S = 0.88e-3
DWELL_S = 5e-6
FIRST_LINE_START_S = 4.57e-3    # start of the first imaging line after the RF centre

# ACDC hardware, as in generation 08.
MAX_CURRENT_A = 3.0
DISABLED_CHANNELS = [9]         # physical outputs, 1..32
TOTAL_CURRENT_LIMIT_A = 30.0
FIELD_LIMIT_HZ = 1000.0
RESPONSE_TAU_S = 133e-6 / np.log(100.0)

# Phase objective (generation 10).
INTRAVOXEL_WEIGHT = 1 / 12
SIGNAL_THRESHOLD = 0.5

# Image objective of generation 08, for comparison.
RUN_IMAGE_OPTIMIZATION = True
IMAGE_ITERATIONS = 300
IMAGE_LEARNING_RATE_A = 0.01
FIELD_GUIDANCE_WEIGHT = 0.05
FIELD_GUIDANCE_SCALE_HZ = 100.0

DEVICE = "auto"                 # "auto", "cuda" or "cpu"; the CPU is fast enough for this notebook

## 2. Numerical phantom

An axial slice through a head-sized object, in the coil frame of the calibration (x: left–right, y: back–front, B0 along z).
Three spherical air cavities lie below the slice. Each creates the exact dipole field of a sphere of air in tissue at 3 T.
Their sizes were chosen so that the off-resonance resembles the map measured in acquisition 029.

The field is known in 3D, so its variation across each voxel is also known **through the slice**. Above the nasal cavity this variation is large: there the signal is lost at TE, as in a real EPI. The last panel shows the fraction of signal left at TE in each voxel without any correction.

In [ ]:
phantom = make_head_phantom(MATRIX, FOV_M, SLICE_Z_M, SLICE_THICKNESS_M, B0_SCALE)
sequence = make_epi_sequence(MATRIX, FOV_M, ECHO_SPACING_S, DWELL_S, FIRST_LINE_START_S)
signal_left = phantom.signal_fraction(sequence.TE_s)     # per voxel, without correction

b0 = phantom.B0_Hz[phantom.support]
table([("Object voxels", phantom.n_voxels),
       ("Voxel size [mm]", " x ".join(f"{1e3 * size:.2f}" for size in phantom.voxel_size_m)),
       ("B0 minimum / maximum [Hz]", f"{b0.min():.0f} / {b0.max():.0f}"),
       ("B0 standard deviation [Hz]", b0.std()),
       ("Largest B0 change through the slice [Hz]", np.abs(phantom.B0_gradient_Hz[..., 2]).max()),
       (f"Voxels keeping more than {SIGNAL_THRESHOLD:.0%} of their signal at TE [%]",
        100 * np.mean(signal_left > SIGNAL_THRESHOLD))])

outside = ~phantom.support
panels = [(phantom.amplitude, "Amplitude after the RF pulse", "gray", 0, None, "a.u."),
          (np.where(outside, np.nan, phantom.B0_Hz), "B0 at the voxel centre", "RdBu_r", -150, 150, "Hz"),
          (np.where(outside, np.nan, phantom.B0_gradient_Hz[..., 2]), "B0 change through the slice", "RdBu_r", -60, 60, "Hz"),
          (phantom.to_image(signal_left), "Signal left at TE, no correction", "gray", 0, 1, "fraction")]
figure, axes = plt.subplots(1, 4, figsize=(17, 4), layout="constrained")
for axis, (image, title, cmap, vmin, vmax, unit) in zip(axes, panels):
    artist = axis.pcolormesh(phantom.X_m * 1e3, phantom.Y_m * 1e3, image, cmap=cmap, vmin=vmin, vmax=vmax,
                             shading="nearest", rasterized=True)
    axis.set(title=title, aspect="equal", xlabel="x [mm]")
    axis.set_facecolor("black" if cmap == "gray" else "#f1f0ec")
    axis.grid(False)
    figure.colorbar(artist, ax=axis, label=unit, shrink=0.85)
axes[0].set_ylabel("y [mm]")
figure.savefig(OUTPUT_DIR / "phantom.png", dpi=150)
plt.show()

## 3. EPI timing, ACDC hardware and coil fields

One current state is played per EPI line, from the start of the line to the start of the next one. Nothing is played before the first imaging line.

The left panel shows the first lines: the readout windows (alternating direction), the echo times $T_n$ and the current states $x_n$.
The right panel shows the matrix $H$ that turns the commands into the running sum of the notes: $H[n, i]$ is the time during which state $i$ has acted before echo $n$. Past lines count for one full echo spacing, the current line for the part already played, future lines for nothing. The amplifier lag only rounds these values slightly.

In [ ]:
hardware = make_hardware(n_channels=32, max_current_A=MAX_CURRENT_A, disabled_channels=DISABLED_CHANNELS,
                         total_current_A=TOTAL_CURRENT_LIMIT_A, field_limit_Hz=FIELD_LIMIT_HZ,
                         response_tau_s=RESPONSE_TAU_S)
calibration = load_calibration(CALIBRATION_PATH)
coil_fields = sample_coil_fields(calibration, phantom.voxel_centres_m(), phantom.voxel_size_m)
if not coil_fields.covered.all():
    raise ValueError(f"{np.count_nonzero(~coil_fields.covered)} phantom voxels lie outside the ACDC calibration; "
                     "move the slice (SLICE_Z_M).")

# The problem "as in the notes": voxel centres, all voxels with the same weight.
# It also provides the diagnostics (phase, residual frequency) used for every method.
notes_problem = PhaseProblem(phantom, coil_fields, sequence, hardware)

table([("Matrix / field of view", f"{MATRIX} x {MATRIX} / {1e3 * FOV_M:.0f} mm"),
       ("Echo spacing [ms]", 1e3 * sequence.echo_spacing_s),
       ("First echo T_1 / echo time TE / last echo [ms]",
        " / ".join(f"{1e3 * t:.2f}" for t in (sequence.echo_time_s[0], sequence.TE_s, sequence.echo_time_s[-1]))),
       ("Active ACDC channels", int(np.count_nonzero(hardware.max_current_A))),
       ("Limits", f"{MAX_CURRENT_A} A per channel, {TOTAL_CURRENT_LIMIT_A} A in total, {FIELD_LIMIT_HZ:.0f} Hz"),
       ("Amplifier time constant [us]", 1e6 * hardware.response_tau_s)])

figure, (left, right) = plt.subplots(1, 2, figsize=(14, 3.8), layout="constrained", width_ratios=(2, 1))
shown = 5
for line in range(shown):
    start, end = 1e3 * sequence.state_start_s[line], 1e3 * sequence.state_end_s[line]
    samples = 1e3 * sequence.time_s[line]
    direction = 1 if sequence.time_s[line, -1] > sequence.time_s[line, 0] else -1
    left.fill_between([samples.min(), samples.max()], 0, direction, color="#2a78d6", alpha=0.35, linewidth=0)
    left.axvline(start, color="#52514e", linewidth=0.8, linestyle=":")
    left.plot(1e3 * sequence.echo_time_s[line], direction, "o", color="#0b0b0b", markersize=6)
    left.annotate(f"$T_{line + 1}$", (1e3 * sequence.echo_time_s[line], direction), textcoords="offset points",
                  xytext=(0, 8 * direction - 4), ha="center", va="bottom" if direction > 0 else "top")
    left.text(0.5 * (start + end), -1.75, f"$x_{line + 1}$", ha="center")
left.axhline(0, color="#52514e", linewidth=0.8)
left.set(xlabel="Time after the RF pulse [ms]", ylabel="Readout direction", ylim=(-2.1, 1.7), yticks=[-1, 0, 1],
         title="First EPI lines: readout windows, echo times and current states")
left.grid(False)
artist = right.imshow(1e3 * notes_problem.current_integral_s[:10, :10], cmap="Blues", vmin=0)
right.set(xlabel="Current state i", ylabel="Echo n", title="H[n, i] (first 10 lines)",
          xticks=range(0, 10, 2), xticklabels=range(1, 11, 2), yticks=range(0, 10, 2), yticklabels=range(1, 11, 2))
right.grid(False)
figure.colorbar(artist, ax=right, label="ms")
figure.savefig(OUTPUT_DIR / "sequence.png", dpi=150)
plt.show()

strongest = np.argsort(-np.abs(coil_fields.maps_Hz_per_A).max(axis=0))[:4]
figure = image_row(phantom, [phantom.to_image(coil_fields.maps_Hz_per_A[:, channel]) for channel in strongest],
                   [f"Channel {channel + 1}" for channel in strongest], "Hz/A", cmap="RdBu_r", vmin=-150, vmax=150,
                   suptitle="Field of four ACDC channels in the slice")
plt.show()

## 4. MRzero simulation without ACDC

MRzero is run once for the phantom in a perfect magnet (the **ideal image**, used as the reference for every comparison) and once with its off-resonance.
The check below compares the magnetisation returned by MRzero with the closed formula
$M_p(t) = m_p\,e^{-t/T_2^*}\,e^{-2\pi i\,(k_x x_p + k_y y_p + B_0 t)}$ on random samples.

In [ ]:
started = time.perf_counter()
simulator = EpiSimulator(phantom, sequence, coil_fields, hardware, device=DEVICE)
print(f"MRzero simulations: {time.perf_counter() - started:.1f} s on {simulator.device}")

# Check MRzero against the closed formula on 200 random ADC samples.
picked = np.random.default_rng(0).choice(simulator.n_samples, 200, replace=False)
support = phantom.support
t = sequence.time_s.ravel()[picked][None, :]
turns = (sequence.kx_mInv.ravel()[picked][None, :] * phantom.X_m[support][:, None]
         + sequence.ky_mInv.ravel()[picked][None, :] * phantom.Y_m[support][:, None]
         + phantom.B0_Hz[support][:, None] * t)
formula = phantom.amplitude[support][:, None] * np.exp(-t / phantom.T2star_s[support][:, None]) * np.exp(-2j * np.pi * turns)
from_mrzero = simulator.magnetization[:, picked].cpu().numpy()
mrzero_error = np.linalg.norm(from_mrzero - formula) / np.linalg.norm(formula)
print(f"MRzero vs closed formula: relative difference {mrzero_error:.1e}")
assert mrzero_error < 5e-3

ideal_magnitude = np.abs(simulator.ideal_image)

def image_error_percent(image):
    # NRMSE of the magnitude image against the ideal image (the measure used by generation 08).
    return 100 * float(np.sqrt(relative_image_error(image, simulator.ideal_image)))

def signal_percent(image):
    # Total signal of the image, relative to the ideal image.
    return 100 * float(np.abs(image).sum() / ideal_magnitude.sum())

n_channels, n_lines = hardware.n_channels, sequence.n_lines
currents = {"No ACDC": np.zeros((n_channels, n_lines))}      # commands of every method, [channels, lines]
images = {"No ACDC": simulator.simulate(currents["No ACDC"])}

figure = image_row(phantom, [ideal_magnitude, np.abs(images["No ACDC"])],
                   ["Ideal image (perfect magnet)", f"EPI without ACDC — error {image_error_percent(images['No ACDC']):.1f} %"],
                   "magnitude [a.u.]", vmin=0, vmax=ideal_magnitude.max())
figure.savefig(OUTPUT_DIR / "baseline.png", dpi=150)
plt.show()

## 5. Phase optimisation

With the matrix $H$ of section 3, the phase of all voxels at all echoes is, in matrix form,

$$\Phi \;=\; 2\pi\,\big(b_0\,T^{\mathsf T} + B\,X\,H^{\mathsf T}\big), \qquad \Psi(X) = \tfrac{1}{P}\lVert\Phi\rVert_F^2 ,$$

with $b_0$ the off-resonance of the voxels [Hz], $B$ the coil fields [Hz/A], $X$ the commands [channel, line] and $T$ the echo times.
$\Psi$ is minimised under the hardware limits by accelerated projected gradient (`phase_optimization.py`).

Two criteria are solved:

- **Voxel centres** — the formula of the introduction, taken literally: every voxel is one point and all voxels count equally.
- **Voxel volume + signal mask** — the same formula with two changes.
  A voxel spans the pixel and the whole 5 mm slice. If the phase changes by $d_x, d_y, d_z$ across it, the mean of $\varphi^2$ over its volume is $\varphi_c^2 + (d_x^2+d_y^2+d_z^2)/12$: the objective then also sees dephasing through the slice (`INTRAVOXEL_WEIGHT`).
  Voxels whose signal is already lost at TE are left out of the mean (`SIGNAL_THRESHOLD`), because their large phase would otherwise dominate $\Psi$.

For each criterion, a **static shim** is also computed: one constant current per channel that minimises the mean squared residual *frequency* $B_0 + B\,I$. It tells how much of the result comes from changing the currents during the readout.

In [ ]:
volume_problem = PhaseProblem(phantom, coil_fields, sequence, hardware,
                              intravoxel_weight=INTRAVOXEL_WEIGHT,
                              voxel_weights=(signal_left > SIGNAL_THRESHOLD).astype(float))
problems = {"voxel centres": notes_problem, "voxel volume + signal mask": volume_problem}

phase_runs = {}
for criterion, problem in problems.items():
    constant = static_shim(problem, hardware, coil_fields)
    currents[f"Static shim, {criterion}"] = np.repeat(constant[:, None], n_lines, axis=1)

    started = time.perf_counter()
    run = optimize_phase(problem, hardware, coil_fields)
    run["seconds"] = time.perf_counter() - started
    phase_runs[criterion] = run
    currents[f"Phase, {criterion}"] = run["currents_A"]

table([(criterion, len(run["objective_history"]) - 1, f"{run['seconds']:.1f}",
        problem.objective(currents["No ACDC"]), problem.objective(currents[f"Static shim, {criterion}"]),
        run["objective_history"][-1])
       for (criterion, problem), run in zip(problems.items(), phase_runs.values())],
      ("Criterion", "Iterations", "Time [s]", "Psi without ACDC [rad²]", "Psi, static shim [rad²]", "Psi, optimised [rad²]"))

The left panel shows the convergence of $\Psi$. The right panel shows the result echo by echo: the RMS phase $\lVert\varphi_n\rVert$ over the object (voxel centres, all voxels), for every set of currents.
Without ACDC the phase grows linearly with the echo time; a perfect correction would stay at zero.

In [ ]:
def method_color(name):
    return METHOD_COLORS.get(name, "#52514e")

figure, (left, right) = plt.subplots(1, 2, figsize=(14, 4), layout="constrained")
for criterion, run in phase_runs.items():
    left.semilogy(run["objective_history"], color=method_color(f"Phase, {criterion}"), label=f"Phase, {criterion}")
left.set(xlabel="Iteration", ylabel="Psi [rad²]", title="Convergence of the phase objective")
left.legend()
for name, commands in currents.items():
    right.plot(1e3 * sequence.echo_time_s, np.sqrt(notes_problem.mean_square_phase(commands)),
               color=method_color(name), label=name, linestyle="--" if name.startswith("Static") else "-")
right.set(xlabel="Echo time T_n [ms]", ylabel="RMS phase over the object [rad]",
          title="Phase at every echo (voxel centres, all voxels)")
right.legend()
figure.savefig(OUTPUT_DIR / "phase_objective.png", dpi=150)
plt.show()

## 6. Reference: the image objective of generation 08

The same phantom is corrected with the objective of generation 08: relative squared error between the simulated EPI magnitude and the target, plus the field guidance, minimised by projected Adam from zero currents.
The target is the ideal image. Every iteration needs one MRzero-based simulation and its gradient.

In [ ]:
image_run = None
if RUN_IMAGE_OPTIMIZATION:
    started = time.perf_counter()
    image_run = optimize_image(simulator, hardware, coil_fields, notes_problem, currents["No ACDC"],
                               iterations=IMAGE_ITERATIONS, learning_rate_A=IMAGE_LEARNING_RATE_A,
                               guidance_weight=FIELD_GUIDANCE_WEIGHT, guidance_scale_Hz=FIELD_GUIDANCE_SCALE_HZ)
    image_run["seconds"] = time.perf_counter() - started
    currents["Image objective (gen. 08)"] = image_run["currents_A"]
    print(f"{IMAGE_ITERATIONS} iterations in {image_run['seconds']:.0f} s on {simulator.device}")

    figure, axis = plt.subplots(figsize=(7, 3.6), layout="constrained")
    axis.plot(100 * np.sqrt(image_run["image_error_history"]), color=method_color("Image objective (gen. 08)"))
    axis.set(xlabel="Iteration", ylabel="Image error [%]", title="Image objective of generation 08: error against the ideal image")
    plt.show()
else:
    print("Image optimisation skipped (RUN_IMAGE_OPTIMIZATION = False).")

## 7. Currents and hardware limits

Commands of the three time-varying solutions (one column per EPI line), then the total current of every method.
The limits are checked on the commands; the real currents, which follow the commands with a lag, are then also inside them.
The last columns repeat the check after rounding to the 16-bit DAC.

In [ ]:
dynamic = [name for name in currents if name.startswith(("Phase", "Image"))]
figure, axes = plt.subplots(1, len(dynamic) + 1, figsize=(5.2 * (len(dynamic) + 1), 5), layout="constrained")
for axis, name in zip(axes, dynamic):
    artist = axis.imshow(currents[name], cmap="RdBu_r", vmin=-MAX_CURRENT_A, vmax=MAX_CURRENT_A,
                         aspect="auto", interpolation="nearest",
                         extent=(0.5, n_lines + 0.5, n_channels + 0.5, 0.5))
    axis.set(title=name, xlabel="EPI line", ylabel="Channel")
    axis.grid(False)
figure.colorbar(artist, ax=axes[len(dynamic) - 1], label="A")
for name, commands in currents.items():
    if name != "No ACDC":
        axes[-1].plot(np.arange(1, n_lines + 1), np.abs(commands).sum(axis=0), color=method_color(name),
                      label=name, linestyle="--" if name.startswith("Static") else "-")
axes[-1].axhline(TOTAL_CURRENT_LIMIT_A, color="#52514e", linewidth=0.8, linestyle=":")
axes[-1].set(title="Total current", xlabel="EPI line", ylabel="Sum of |current| over the channels [A]",
             ylim=(0, 1.08 * TOTAL_CURRENT_LIMIT_A))
axes[-1].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.2))
figure.savefig(OUTPUT_DIR / "currents.png", dpi=150)
plt.show()

quantized = {name: quantize_to_dac(commands, hardware) for name, commands in currents.items()}
limit_rows = []
for name in currents:
    if name == "No ACDC":
        continue
    before = check_limits(currents[name], hardware, coil_fields.maps_Hz_per_A, coil_fields.gradients_Hz_per_A)
    after = check_limits(quantized[name], hardware, coil_fields.maps_Hz_per_A, coil_fields.gradients_Hz_per_A)
    all_met = lambda report: all(value for key, value in report.items() if key.endswith("met"))
    limit_rows.append((name, before["max |current| [A]"], before["max summed |current| [A]"],
                       before["max |ACDC field| [Hz]"], all_met(before), all_met(after)))
    assert all_met(before) and all_met(after), f"{name}: a hardware limit is exceeded."
table(limit_rows, ("Method", "Max current [A]", "Max total current [A]", "Max ACDC field [Hz]",
                   "Limits met", "Limits met after DAC"))

## 8. Simulated images

Every set of currents is played in the MRzero-based simulation (with the amplifier response). The images are compared with the ideal image.

- **Image error**: NRMSE of the magnitude image against the ideal image (lower is better).
- **Signal**: sum of the image magnitude relative to the ideal image; signal lost by dephasing lowers it.
- **Psi**: the phase objective of the introduction (voxel centres, all voxels), for reference.
- **Residual frequency**: RMS over the object and the readout of the frequency left between consecutive echoes.

In [ ]:
for name, commands in currents.items():
    images[name] = simulator.simulate(commands)

results = {}
for name, commands in currents.items():
    results[name] = {
        "image_error_percent": image_error_percent(images[name]),
        "image_error_after_DAC_percent": image_error_percent(simulator.simulate(quantized[name])),
        "signal_percent": signal_percent(images[name]),
        "Psi_rad2": notes_problem.objective(commands),
        "residual_frequency_rms_Hz": float(np.sqrt(np.mean(notes_problem.residual_frequency_Hz(commands)**2))),
        "mean_total_current_A": float(np.abs(commands).sum(axis=0).mean()),
    }
table([(name, f"{r['image_error_percent']:.1f}", f"{r['image_error_after_DAC_percent']:.1f}", f"{r['signal_percent']:.1f}",
        f"{r['Psi_rad2']:.0f}", f"{r['residual_frequency_rms_Hz']:.1f}", f"{r['mean_total_current_A']:.1f}")
       for name, r in results.items()],
      ("Method", "Image error [%]", "Image error after DAC [%]", "Signal [%]", "Psi [rad²]",
       "Residual frequency [Hz]", "Mean total current [A]"))

names = list(currents)
figure = image_row(phantom, [ideal_magnitude] + [np.abs(images[name]) for name in names],
                   ["Ideal"] + [f"{name}\nerror {results[name]['image_error_percent']:.1f} %" for name in names],
                   "magnitude [a.u.]", vmin=0, vmax=ideal_magnitude.max())
figure.savefig(OUTPUT_DIR / "images.png", dpi=150)
plt.show()
limit = 0.5 * ideal_magnitude.max()
figure = image_row(phantom, [np.abs(images[name]) - ideal_magnitude for name in names], names,
                   "image − ideal [a.u.]", cmap="RdBu_r", vmin=-limit, vmax=limit,
                   suptitle="Difference with the ideal image (blue: signal missing, red: signal in excess)")
figure.savefig(OUTPUT_DIR / "image_differences.png", dpi=150)
plt.show()

## 9. Why the methods differ

Two maps explain the images, both evaluated at the echo time TE:

- the **phase change through the slice**, in turns: when it reaches one turn the spins of the voxel cancel and the signal is lost. The coils create such a variation themselves when they are driven hard;
- the **residual frequency** around TE: it shifts the signal along the phase-encoding direction (y), by one pixel for every $1/(N\,\Delta T_{echo})$ Hz.

The last table tests how much each solution relies on *changing* the currents during the readout: every command is replaced by its mean over the lines and the image is simulated again.

In [ ]:
centre = n_lines // 2
through_slice, frequency = [], []
for name, commands in currents.items():
    through_slice.append(phantom.to_image(notes_problem.phase_across_voxel_rad(commands, axis=2)[:, centre] / (2 * np.pi)))
    frequency.append(phantom.to_image(notes_problem.residual_frequency_Hz(commands)[:, centre]))
figure = image_row(phantom, through_slice, names, "turns", cmap="RdBu_r", vmin=-2, vmax=2,
                   suptitle="Phase change through the slice at TE (signal lost at ±1 turn)")
figure.savefig(OUTPUT_DIR / "through_slice_phase.png", dpi=150)
plt.show()
pixel_Hz = 1 / (n_lines * sequence.echo_spacing_s)
figure = image_row(phantom, frequency, names, "Hz", cmap="RdBu_r", vmin=-100, vmax=100,
                   suptitle=f"Residual frequency at TE (one pixel of shift along y per {pixel_Hz:.1f} Hz)")
figure.savefig(OUTPUT_DIR / "residual_frequency.png", dpi=150)
plt.show()

rows = []
for name in dynamic:
    averaged = np.repeat(currents[name].mean(axis=1, keepdims=True), n_lines, axis=1)
    results[name]["image_error_time_averaged_percent"] = image_error_percent(simulator.simulate(averaged))
    rows.append((name, f"{results[name]['image_error_percent']:.1f}",
                 f"{results[name]['image_error_time_averaged_percent']:.1f}"))
table(rows, ("Method", "Image error [%]", "Image error with time-averaged currents [%]"))

## 10. Save the results

The currents, images and the summary table are saved in the session folder. Each run creates a new folder.

In [ ]:
arrays = {"X_m": phantom.X_m, "Y_m": phantom.Y_m, "amplitude": phantom.amplitude, "B0_Hz": phantom.B0_Hz,
          "B0_gradient_Hz": phantom.B0_gradient_Hz, "ideal_image": simulator.ideal_image,
          "echo_time_s": sequence.echo_time_s, "state_start_s": sequence.state_start_s,
          "state_end_s": sequence.state_end_s}
for index, name in enumerate(currents):
    arrays[f"currents_A_{index}"] = currents[name]
    arrays[f"image_{index}"] = images[name]
for criterion, run in phase_runs.items():
    arrays["Psi_history_" + criterion.replace(" ", "_").replace("+", "and")] = run["objective_history"]
if image_run is not None:
    arrays["image_error_history"] = image_run["image_error_history"]
np.savez_compressed(OUTPUT_DIR / "mrzero10_phase_optimization.npz", **arrays)

parameters = {name: value for name, value in globals().items()
              if name.isupper() and isinstance(value, (int, float, str, list, bool))}
summary = {"notebook": "10_acdc_phase_optimization_MRzero.ipynb", "generation": "10",
           "parameters": parameters, "methods": list(currents), "results": results,
           "device": str(simulator.device), "MRzero_vs_formula_relative_difference": float(mrzero_error),
           "phase_optimization_seconds": {criterion: run["seconds"] for criterion, run in phase_runs.items()},
           "image_optimization_seconds": None if image_run is None else image_run["seconds"]}
(OUTPUT_DIR / "mrzero10_phase_optimization.json").write_text(json.dumps(summary, indent=2))
print("Saved in", OUTPUT_DIR)

## Reading the results

- The table of section 8 is the answer to "is it better to optimise the phase?" for this phantom: compare the image error of the two *Phase* rows with *No ACDC* and with *Image objective (gen. 08)*.
- $\Psi$ and the image error do not rank the methods in the same order. $\Psi$ measures phase in radians wherever the object is, including where the phase is many turns and cannot be corrected by the coils; the image only depends on phase differences inside a voxel and between echoes.
- If a *Phase* row and the *Static shim* row of the same criterion give the same image error, the phase objective did not gain anything from changing the currents during the readout.
- The phase objective assumes that the B0 map is unwrapped and has the correct absolute frequency.

The result of the first run is summarised in `README.md`.